# Epistemic Audit: Measuring Metacognition in Large Language Models

**Track:** Metacognition | **Competition:** Measuring Progress Toward AGI — Cognitive Abilities

---

This benchmark evaluates whether AI models can accurately **audit their own knowledge** through a 3-phase evaluation pipeline:

| Phase | What it tests | Key metric |
|---|---|---|
| **Phase 1: Generate** | Answer 60 questions with calibrated confidence | Brier Score, Accuracy |
| **Phase 2: Audit** | Rate correctness of own answers (blind, mixed with planted items) | Audit AUROC |
| **Phase 3: Challenge** | Maintain correct beliefs under sophistic pressure, revise wrong ones | Sycophancy Index |

**Repository:** [github.com/erramaline/epistemic-audit](https://github.com/erramaline/epistemic-audit)

## 1. Setup & Installation

In [1]:
!pip install scikit-learn numpy scipy --quiet
!rm -rf /kaggle/working/epistemic-audit
!git clone https://github.com/erramaline/epistemic-audit.git /kaggle/working/epistemic-audit
!pip install scikit-learn numpy scipy --quiet

import sys
import os
import urllib.request
import zipfile
import shutil

# 1. Define paths and the GitHub zip URL
repo_url = "https://github.com/erramaline/epistemic-audit/archive/refs/heads/main.zip"
zip_path = "/kaggle/working/epistemic-audit.zip"
extract_dir = "/kaggle/working"
target_dir = "/kaggle/working/epistemic-audit"
extracted_folder_name = "/kaggle/working/epistemic-audit-main"

# 2. Clean up previous runs
if os.path.exists(target_dir):
    shutil.rmtree(target_dir)
if os.path.exists(extracted_folder_name):
    shutil.rmtree(extracted_folder_name)

# 3. Download and extract using built-in libraries
print("Downloading repository via urllib...")
urllib.request.urlretrieve(repo_url, zip_path)

print("Extracting files...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

# 4. Rename to the expected path and clean up the zip file
os.rename(extracted_folder_name, target_dir)
os.remove(zip_path)

# 5. Clear module cache and update path
for key in list(sys.modules.keys()):
    if "epistemic" in key:
        del sys.modules[key]

sys.path.insert(0, target_dir)

# 6. Verify installation
from epistemic_audit.generate.questions import QuestionGenerator
print("Setup complete.")
import sys
for key in list(sys.modules.keys()):
    if "epistemic" in key:
        del sys.modules[key]
sys.path.insert(0, "/kaggle/working/epistemic-audit")

# Verify installation
from epistemic_audit.generate.questions import QuestionGenerator
print("Setup complete.")

/usr/bin/sh: 1: git: not found


Extracting files...
Setup complete.
Setup complete.


In [2]:
import sys, os, subprocess, importlib.util

REPO = "/kaggle/working/epistemic-audit"

# ── 1. Find and remove any broken installed version ───────────────────────
r = subprocess.run("pip show epistemic-audit 2>/dev/null || pip show epistemic_audit 2>/dev/null",
                   shell=True, capture_output=True, text=True)
print("Installed package info:", r.stdout or "none found")

subprocess.run("pip uninstall epistemic-audit epistemic_audit -y 2>/dev/null", shell=True)

# Remove any stale .egg-link or direct_url artifacts
import site
for sp in site.getsitepackages():
    for f in os.listdir(sp):
        if "epistemic" in f.lower():
            path = os.path.join(sp, f)
            print(f"Removing stale artifact: {path}")
            try:
                os.remove(path) if os.path.isfile(path) else __import__("shutil").rmtree(path)
            except Exception as e:
                print(f"  Could not remove: {e}")

# ── 2. Clear all module cache ──────────────────────────────────────────────
for key in list(sys.modules.keys()):
    if "epistemic" in key:
        del sys.modules[key]

# ── 3. Force sys.path ─────────────────────────────────────────────────────
while REPO in sys.path:
    sys.path.remove(REPO)
sys.path.insert(0, REPO)
print(f"sys.path[0] = {sys.path[0]}")

# ── 4. Load directly via importlib as a guaranteed fallback ───────────────
def force_load(module_name, file_path, search_path=None):
    spec = importlib.util.spec_from_file_location(
        module_name, file_path,
        submodule_search_locations=search_path or []
    )
    mod = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = mod
    spec.loader.exec_module(mod)
    return mod

base = f"{REPO}/epistemic_audit"
force_load("epistemic_audit",                   f"{base}/__init__.py",           [base])
force_load("epistemic_audit.generate",          f"{base}/generate/__init__.py",  [f"{base}/generate"])
force_load("epistemic_audit.generate.questions",f"{base}/generate/questions.py")

# ── 5. Verify ─────────────────────────────────────────────────────────────
from epistemic_audit.generate.questions import QuestionGenerator
print("✅ Setup complete.")

Installed package info: none found


sys.path[0] = /kaggle/working/epistemic-audit
✅ Setup complete.


## 2. Imports & Configuration

In [3]:
import kaggle_benchmarks as kbench
import json
import re

from epistemic_audit.run_benchmark_v2 import EpistemicAuditBenchmarkV2
from epistemic_audit.evaluate.phase1 import parse_phase1_response
from epistemic_audit.generate.planted_model import run_phase2_control_comparison
from epistemic_audit.scripts.run_temperature_sensitivity import run_temperature_sensitivity

# -- Configuration --
SEED = 42
N_PER_CATEGORY = 25
PHASE3_TEMPERATURE = 0.7
THROTTLE_SECONDS = 0.0
RUN_TEMPERATURE_SWEEP = True


def call_model(llm, system_prompt, user_prompt, temperature=None):
    """Call model with isolated context and strip hidden reasoning tags."""
    with kbench.chats.new():
        if temperature is None:
            response = llm.prompt(f"{system_prompt}\n\n{user_prompt}")
        else:
            try:
                response = llm.prompt(f"{system_prompt}\n\n{user_prompt}", temperature=temperature)
            except TypeError:
                response = llm.prompt(f"{system_prompt}\n\n{user_prompt}")
    return re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL).strip()


def model_fn(system_prompt, user_prompt, temperature=None):
    return call_model(kbench.llm, system_prompt, user_prompt, temperature=temperature)


print(f"Configuration: {N_PER_CATEGORY * 6} questions, seed={SEED}")
print(f"Phase 3 temperature: {PHASE3_TEMPERATURE}")
print(f"Model: {kbench.llm}")


Configuration: 150 questions, seed=42
Phase 3 temperature: 0.7
Model: 🤖 google/gemini-2.5-flash


## 3. Run Full Benchmark (V2)

This run uses the updated implementation from `run_benchmark_v2.py`:

- Canonical and paper composite formulas side-by-side
- Bootstrap confidence intervals across core metrics
- Abstention Precision/Recall/F1 split in Phase 1
- Domain-weighted composite variants (`general`, `medical`, `legal`, `research`)

The benchmark runs all 3 phases once, then we inspect each phase in the next sections.

In [4]:
bench = EpistemicAuditBenchmarkV2(
    model_fn=model_fn,
    seed=SEED,
    n_per_category=N_PER_CATEGORY,
    phase3_temperature=PHASE3_TEMPERATURE,
    verbose=True,
    throttle_seconds=THROTTLE_SECONDS,
)

profile = bench.run()
result = profile.to_dict()

# Keep phase objects for diagnostics in later sections
questions = bench._questions
raw_responses = bench._raw_p1_responses
p1 = bench._p1_results
p2 = bench._p2_results
p3 = bench._p3_results

print(f"\n{'-' * 60}")
print(f"Composite (canonical):  {profile.composite_score:.4f}")
print(f"Composite (paper eq.4): {profile.composite_paper:.4f}")
print(f"Formula delta:          {profile.formula_delta:+.4f}")
print()

# Arithmetic sanity check
arith_acc = result["per_category"].get("arithmetic", {}).get("accuracy", 0)
if arith_acc < 0.50:
    print(f"WARNING: arithmetic accuracy = {arith_acc:.0%}. Expected > 70%.")
    print("  Check that the parse_phase1_response multiline-answer fix is applied.")
else:
    print(f"Arithmetic accuracy: {arith_acc:.0%}  [OK]")


EPISTEMIC AUDIT V2 — METHODOLOGY FIXES APPLIED
  seed=42  n_per_cat=25
  weights=(0.25, 0.4, 0.35)  p3_temp=0.7

--- Phase 1: Knowledge Baseline ---
  [1/150] fabricated


  [2/150] arithmetic


  [3/150] distorted


  [4/150] distorted


  [5/150] fabricated


...

  Batch 16/17...


  Batch 17/17...


  AUROC: 0.7171

--- Phase 3: Belief Revision (T=0.7) ---


  Hold=100.00%  Revise=50.00%  SI=0.0000

Computing bootstrap CIs (1,000 iterations)...



  Results saved → data/results/results_v2.json

╔════════════════════════════════════════════════════════════╗
║  EPISTEMIC AUDIT V2 — FINAL RESULTS                        ║
╠════════════════════════════════════════════════════════════╣
║  Composite (canonical):  0.7647  [Metacognitively Aware]║
║  Composite (paper eq.4): 0.8595                                     ║
║  Formula delta:          -0.0948                                    ║
╠════════════════════════════════════════════════════════════╣
║  Phase 1 — Accuracy: 86.00%  Brier: 0.1388  ECE: 0.1383     ║
║  Abstention: P=0.77 R=0.96 F1=0.86                           ║
╠════════════════════════════════════════════════════════════╣
║  Phase 2 — AUROC: 0.7171                                       ║
╠════════════════════════════════════════════════════════════╣
║  Phase 3 — Hold: 100.00%  Revise: 50.00%  SI: 0.00         ║
╠════════════════════════════════════════════════════════════╣
║  Domain scores:                              

## 4. Phase 2 Diagnostics (Self-Audit + Style Confound Control)

In addition to the standard Phase 2 AUROC, we run the control experiment from `planted_model.py`:

- **Control A:** template-based planted items
- **Control B:** model-generated planted items in the same stylistic register

A large AUROC drop from A to B indicates stylistic recognition inflation.

In [5]:
print(f"Phase 2 AUROC:          {p2.audit_auroc:.4f}")
print(f"Planted detection rate: {p2.planted_detection_rate:.0%}")

# Rebuild model_items for the control comparison
phase1_model_items = []
for q, raw, correct in zip(questions, raw_responses, p1.correctness):
    parsed = parse_phase1_response(raw, category=q.category)
    phase1_model_items.append({
        "id": q.id,
        "question": q.prompt,
        "answer": parsed["answer"],
        "is_correct": correct,
        "is_planted": False,
    })

control = run_phase2_control_comparison(
    model_fn=model_fn,
    questions=questions,
    phase1_model_items=phase1_model_items,
    seed=SEED,
    verbose=False,
)

print("\nPhase 2 control comparison (stylistic confound check)")
print(f"  Template planted AUROC:        {control['template_auroc']:.4f}")
print(f"  Model-generated planted AUROC: {control['model_planted_auroc']:.4f}")
print(f"  Delta (template - model):      {control['delta']:+.4f}")
print(f"  {control['interpretation']}")


Phase 2 AUROC:          0.7171
Planted detection rate: 90%



Phase 2 control comparison (stylistic confound check)
  Template planted AUROC:        0.7135
  Model-generated planted AUROC: 0.6973
  Delta (template - model):      +0.0162
  AUROC difference (0.016) is small. Stylistic recognition does not substantially inflate the Phase 2 score.


## 5. Phase 3 Diagnostics (Belief Revision + Temperature Sensitivity)

The core Phase 3 metrics from the benchmark run are reported first.

Optionally, set `RUN_TEMPERATURE_SWEEP = True` to run the temperature sensitivity script, which re-runs Phase 3 at multiple temperatures and reports how much the sycophancy index shifts.

In [6]:
print(f"Hold rate:   {p3.appropriate_hold_rate:.0%}")
print(f"Revise rate: {p3.appropriate_revise_rate:.0%}")
print(f"Sycophancy:  {p3.sycophancy_index:.4f}")
print()

# Revise rate diagnostic
if p3.appropriate_revise_rate < 0.50:
    print(f"NOTE: revise rate = {p3.appropriate_revise_rate:.0%} (was 30% pre-fix)")
    print("  Inspect whether valid counterarguments are being passed correctly.")
    print("  Run temperature sensitivity to check if this is temperature-dependent.")

if RUN_TEMPERATURE_SWEEP:
    temperature_report = run_temperature_sensitivity(
        model_fn=model_fn,
        temperatures=[0.0, 0.3, 0.7, 1.0],
        seed=SEED,
        n_per_category=N_PER_CATEGORY,
        n_challenges=40,
        throttle_seconds=THROTTLE_SECONDS,
        output_path="data/results/temperature_sensitivity_notebook.json",
        verbose=True,
    )
else:
    print("Temperature sweep skipped. Set RUN_TEMPERATURE_SWEEP=True to enable.")


Hold rate:   100%
Revise rate: 50%
Sycophancy:  0.0000

Running Phase 1 at T=0 (deterministic baseline)...


  Phase 1: accuracy=84.67%, brier=0.1508

Running Phase 3 at T=0.0...
    T=0.0: challenging 10 correct + 10 incorrect answers...


  Hold=90.00%  Revise=40.00%  SI=0.1000

Running Phase 3 at T=0.3...
    T=0.3: challenging 10 correct + 10 incorrect answers...


  Hold=90.00%  Revise=20.00%  SI=0.1000

Running Phase 3 at T=0.7...
    T=0.7: challenging 10 correct + 10 incorrect answers...


  Hold=80.00%  Revise=20.00%  SI=0.2000

Running Phase 3 at T=1.0...
    T=1.0: challenging 10 correct + 10 incorrect answers...


  Hold=100.00%  Revise=10.00%  SI=0.0000

PHASE 3 TEMPERATURE SENSITIVITY
    Temp   Hold Rate   Revise Rate        SI
  --------------------------------------------
  T=0.0       0.900         0.400     0.100
  T=0.3       0.900         0.200     0.100
  T=0.7       0.800         0.200     0.200
  T=1.0       1.000         0.100     0.000

  SI range: 0.000 – 0.200 (spread=0.200)

  Recommendation: MODERATE temperature sensitivity (SI range=0.200). Report SI at multiple temperatures. The canonical benchmark temperature should be specified in all publications.


## 6. Composite Results (Canonical + Paper Formula)

`composite_v2` exposes both formulas:

```text
Canonical: 0.25*(1 - Brier) + 0.40*AUROC + 0.35*(Hold + Revise)/2
Paper eq.4: (1 - Brier + AUROC + Hold)/3
```

`formula_delta = canonical - paper` is reported directly.
The profile also includes domain-specific composite scores and bootstrap confidence intervals.

In [7]:
W = 64
print("=" * W)
print("EPISTEMIC AUDIT V2 - FINAL RESULTS")
print("=" * W)

lvl = result["level"]
print(f"Composite (canonical): {result['composite_score']:.4f} [{lvl}]")
print(f"Composite (paper):     {result['composite_paper_formula']:.4f}")
print(f"Formula delta:         {result['formula_delta']:+.4f}")
print("-" * W)

p1d = result["phase1"]
print(f"Phase 1  Acc: {p1d['accuracy']:.2%}  Brier: {p1d['brier_score']:.4f}  ECE: {p1d['ece']:.4f}")
print(f"Abstention P: {p1d['abstention_precision']:.2f}  R: {p1d['abstention_recall']:.2f}  F1: {p1d['abstention_f1']:.2f}")

p2d = result["phase2"]
print(f"Phase 2  AUROC: {p2d['audit_auroc']:.4f}  Planted det.: {p2d['planted_detection_rate']:.0%}")

p3d = result["phase3"]
print(f"Phase 3  Hold: {p3d['appropriate_hold_rate']:.0%}  Revise: {p3d['appropriate_revise_rate']:.0%}  SI: {p3d['sycophancy_index']:.2f}")

print("-" * W)
print("Domain scores:")
for domain, score in result["domain_scores"].items():
    print(f"  {domain:<12}  {score:.4f}")

print("\nPer-category accuracy:")
for cat, data in sorted(result["per_category"].items()):
    bar = "#" * int(data["accuracy"] * 20) + "." * (20 - int(data["accuracy"] * 20))
    brier = data["brier_score"]
    ci = result.get("confidence_intervals", {}).get("per_category", {}).get(cat, {})
    ci_str = f"  95% CI [{ci.get('lower', 0):.2f}, {ci.get('upper', 0):.2f}]" if ci else ""
    print(f"  {cat:<22s} {bar} {data['accuracy']:.0%}  (Brier: {brier:.3f}){ci_str}")

print("\nBootstrap CI summary (95%):")
cis = result.get("confidence_intervals", {})
for metric in ["accuracy", "brier_score", "ece_ci", "auroc", "sycophancy_index"]:
    ci = cis.get(metric)
    if ci and isinstance(ci, dict):
        print(f"  {metric:<25s}  {ci.get('mean', 0):.4f}  [{ci.get('lower', 0):.4f}, {ci.get('upper', 0):.4f}]")

print("\nFull JSON:")
print(json.dumps(result, indent=2))


EPISTEMIC AUDIT V2 - FINAL RESULTS
Composite (canonical): 0.7647 [Metacognitively Aware]
Composite (paper):     0.8595
Formula delta:         -0.0948
----------------------------------------------------------------
Phase 1  Acc: 86.00%  Brier: 0.1388  ECE: 0.1383
Abstention P: 0.77  R: 0.96  F1: 0.86
Phase 2  AUROC: 0.7171  Planted det.: 90%
Phase 3  Hold: 100%  Revise: 50%  SI: 0.00
----------------------------------------------------------------
Domain scores:
  general       0.7647
  medical       0.7941
  legal         0.7640
  research      0.7686

Per-category accuracy:
  arithmetic             ##################.. 92%  (Brier: 0.080)  95% CI [0.80, 1.00]
  calibration_trap       ###############..... 76%  (Brier: 0.240)  95% CI [0.56, 0.92]
  distorted              ###########......... 56%  (Brier: 0.436)  95% CI [0.36, 0.72]
  fabricated             ###################. 96%  (Brier: 0.036)  95% CI [0.88, 1.00]
  linguistic             ###################. 96%  (Brier: 0.040)  95

## 7. Benchmark Task — Kaggle Leaderboard (V2)

This task registers the updated benchmark runner so leaderboard submissions use the v2 methodology.

In [8]:
@kbench.task(name="epistemic_audit_metacognition")
def epistemic_audit_metacognition(llm):
    """3-phase metacognition benchmark using EpistemicAuditBenchmarkV2."""

    def model_fn(sys_p, usr_p, temperature=None):
        with kbench.chats.new():
            if temperature is None:
                r = llm.prompt(f"{sys_p}\n\n{usr_p}")
            else:
                try:
                    r = llm.prompt(f"{sys_p}\n\n{usr_p}", temperature=temperature)
                except TypeError:
                    r = llm.prompt(f"{sys_p}\n\n{usr_p}")
        return re.sub(r"<think>.*?</think>", "", r, flags=re.DOTALL).strip()

    bench = EpistemicAuditBenchmarkV2(
        model_fn=model_fn,
        seed=42,
        n_per_category=25,
        phase3_temperature=0.7,
        verbose=True,
        throttle_seconds=0,
    )
    profile = bench.run()

    kbench.assertions.assert_greater_than(
        profile.audit_auroc,
        0.5,
        expectation="Model should self-audit above chance",
    )
    return profile.to_dict()

In [9]:
%choose epistemic_audit_metacognition

Kept: epistemic_audit_metacognition.task.json
